In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [3]:
#Load the data set
df=pd.read_csv('train.csv')
#Add the new interesting feature we have found
df['square']=df['loan_int_rate']**2+df['loan_percent_income']**2

In [4]:
df.head()

,id,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_status,square
0,0,37,35000,RENT,0.0,EDUCATION,B,6000,11.49,0.17,N,14,0,132.0490
1,1,22,56000,OWN,6.0,MEDICAL,C,4000,13.35,0.07,N,2,0,178.2274
2,2,29,28800,OWN,8.0,PERSONAL,A,6000,8.90,0.21,N,10,0,79.2541
3,3,30,70000,RENT,14.0,VENTURE,B,12000,11.11,0.17,N,5,0,123.4610
4,4,22,60000,RENT,2.0,MEDICAL,A,6000,6.92,0.10,N,3,0,47.8964


In [5]:
#Let's sstart with a naive kNearest nhbd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [6]:
list(df.columns)

['id',
 'person_age',
 'person_income',
 'person_home_ownership',
 'person_emp_length',
 'loan_intent',
 'loan_grade',
 'loan_amnt',
 'loan_int_rate',
 'loan_percent_income',
 'cb_person_default_on_file',
 'cb_person_cred_hist_length',
 'loan_status',
 'square']

In [7]:
df=df.drop('id', axis=1)
df=pd.get_dummies(df, drop_first=True, dtype=int)

In [8]:
df.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,loan_status,square,person_home_ownership_OTHER,...,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G,cb_person_default_on_file_Y
0,37,35000,0.0,6000,11.49,0.17,14,0,132.0490,0,...,0,0,0,1,0,0,0,0,0,0
1,22,56000,6.0,4000,13.35,0.07,2,0,178.2274,0,...,1,0,0,0,1,0,0,0,0,0
2,29,28800,8.0,6000,8.90,0.21,10,0,79.2541,0,...,0,1,0,0,0,0,0,0,0,0
3,30,70000,14.0,12000,11.11,0.17,5,0,123.4610,0,...,0,0,1,1,0,0,0,0,0,0
4,22,60000,2.0,6000,6.92,0.10,3,0,47.8964,0,...,1,0,0,0,0,0,0,0,0,0


In [9]:
##### Let's start with a naif k-Closest Nhbd.

In [10]:
features=[x for x in list(df.columns) if x not in['loan_status']]
#n = [x for x in l if x not in m]
#df.columns
print('loan_status' in features)

False


In [11]:
df_train, df_val=train_test_split(df, random_state=42, shuffle=True, test_size=.2, stratify=df['loan_status'])

In [12]:
knn_pipe = Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier(5))])

knn_pipe.fit(df_train[features],
           df_train.loan_status)

Pipeline(steps=[('scale', StandardScaler()), ('knn', KNeighborsClassifier())])

In [13]:
knn_pipe.predict(df_train[features])

array([0, 1, 0, ..., 0, 0, 0])

In [14]:
pred=knn_pipe.predict(df_train[features])

In [15]:
def accuracy(true, predicted):
    return np.sum(true==predicted)/len(predicted)

In [16]:
#print(len(pred), df_train.shape[0])
accuracy(df_train.loan_status,pred)

np.float64(0.947523233012192)

In [17]:
pred_val=knn_pipe.predict(df_val[features])

In [18]:
len(pred_val)

11729

In [19]:
accuracy(df_val.loan_status,pred_val)

np.float64(0.9322192855315884)

In [20]:
###Let's try logistic regression and see how it goes
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression(penalty=None)

In [21]:
log_reg_pipe = Pipeline([('scale', StandardScaler()), ('log_reg',log_reg)])
log_reg_pipe.fit(df_train[features],
           df_train['loan_status'])

Pipeline(steps=[('scale', StandardScaler()),
                ('log_reg', LogisticRegression(penalty=None))])

In [22]:
log_reg.coef_

array([[ 0.02297611, -0.21291397, -0.11566202, -0.37226679,  0.80925976,
         1.30661879,  0.00218591, -0.43572958,  0.02750608, -0.66433966,
         0.55903119, -0.3524366 ,  0.03222615, -0.09780956, -0.23273888,
        -0.44036113,  0.00201218,  0.00309829,  0.773855  ,  0.34114242,
         0.12796048,  0.08648922,  0.02901097]])

In [26]:
loan_status_prob = log_reg.predict_proba(df_train[features])[:,1]
#loan_status_pred = 1*(loan_status_prob >= .7)
## assign the value based on the cutoff
#y_pred = 1*(y_prob >= .7)


In [28]:
np.sum(loan_status_pred)

np.int64(2660)

In [ ]:
df_train.values

In [ ]:
type(df_train.loan_status)

In [ ]:
df_train[features]

In [ ]:
log_reg.predict_proba(df_train[features])